<div style="border-left:14px solid #708090;background:#ffffff;padding:70px 40px;border-radius:4px"><div style="font:600 20px sans-serif;color:#708090;letter-spacing:.08em">WELCOME</div><div style="font:700 48px sans-serif;margin-top:10px;color:#1f2430">Start here</div><div style="font:22px sans-serif;margin-top:16px;color:#1f2430">How this session works, and a check that your setup runs.</div></div>

**Google Colab version: the whole workshop in one notebook.** Run the setup cells below once, from top to bottom (about two minutes). They get the workshop code, so that every path is the same as on the workshop server, and connect your Google Drive for the data of part 3. Then go on with the parts in order. If Colab restarts the runtime, run the setup cells again.

In [ ]:
# Colab setup: run this cell first.
import os
import subprocess
import sys

if not os.path.exists("/content/DH-Workshop"):
    subprocess.run(["git", "clone", "-q", "https://github.com/laurajul/dh-workshop-dsi.git", "/content/DH-Workshop"], check=True)
os.chdir("/content/DH-Workshop/notebooks")
sys.path.insert(0, os.getcwd())
print("ready in", os.getcwd())

In [ ]:
%pip install -q ftfy regex git+https://github.com/openai/CLIP.git

## The shared data folder (Google Drive)

The collection (metadata and CLIP vectors) is in a shared Google Drive folder.

1. Run the first cell below and allow access to your Drive.
2. Open the folder with the link that the second cell shows. At the top, click the folder name, then **Organise, Add shortcut, My Drive**.
3. Run the second cell. It links the folder to `~/data`, the same path as on the workshop server.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import ipywidgets as widgets

SHARED_FOLDER_URL = "PASTE-THE-LINK-TO-THE-SHARED-GOOGLE-DRIVE-FOLDER-HERE"     # <-- the instructor puts the link here
FOLDER_NAME = "DSI-Workshop"        # the name of the folder in your My Drive after adding the shortcut

display(widgets.HTML(f"<a href='{SHARED_FOLDER_URL}' target='_blank' style='font:16px sans-serif'>{SHARED_FOLDER_URL}</a>"))

src = f"/content/drive/MyDrive/{FOLDER_NAME}"
link = os.path.expanduser("~/data")
if os.path.islink(link) or os.path.exists(link):
    print(link, "is already there, leaving it as it is.")
elif not os.path.exists(os.path.join(src, "metadata.csv")):
    print("metadata.csv not found in", src, "- add the shortcut to My Drive (step 2) and check FOLDER_NAME.")
else:
    os.symlink(src, link)
    print("linked", src, "->", link)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#708090"><span>WELCOME · 1 / 4</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #708090;margin:6px 0 0 0">

# Computational Approaches to Cultural Heritage Image Collections

This workshop is about finding, reading and working with digitised cultural heritage **programmatically**. Drawing on the idea of *distant reading*, it asks how algorithms can "read" collections of images and their descriptions at scale, and what they miss.

The path through the workshop:

1. **The IIIF standards** (notebook 1): a shared standard that lets any viewer show images from any institution, at any size and detail.
2. **The Europeana corpus** (notebook 2): an aggregator with millions of descriptions from thousands of institutions, and an API. We treat it as a *dataset*, to find out what is inside.
3. **Semantic search** (notebook 3): a model (CLIP) that lets you search images by describing them in words or with an image, and what changes when you search with a part of an image instead of the whole.
4. **A small task, then short presentations** (final task): each of you shows what you found, then we conclude.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#708090"><span>WELCOME · 2 / 4</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #708090;margin:6px 0 0 0">

## Working in a notebook

- A notebook is a sequence of **cells**. Text cells explain; code cells run Python.
- Click a cell and press **Shift+Enter** to run it. The result appears below it.
- Run cells **in order, from the top**. Later cells use what earlier ones defined.
- Select the kernel **DH Workshop** (top right). If a cell fails with a `NameError`, run the cells above it first.
- You cannot break anything. If a notebook gets into a muddle, use **Kernel → Restart Kernel**, then run from the top.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#708090"><span>WELCOME · 4 / 4</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #708090;margin:6px 0 0 0">

## Check your setup

Run the cell below. Each line should start with ✓. If something does not, tell the instructor now.

In [ ]:
import importlib, platform, sys
from pathlib import Path

def check(label, ok, detail=""):
    print(("✓" if ok else "✗"), label, detail)

check("Python", sys.version_info >= (3, 10), platform.python_version())
for pkg in ["requests", "PIL", "ipywidgets", "numpy", "torch", "clip"]:
    try:
        importlib.import_module(pkg)
        check(f"package {pkg}", True)
    except Exception as e:
        check(f"package {pkg}", False, f"({type(e).__name__})")

import requests
for name, url in [("IIIF server (iiif.io)", "https://iiif.io/api/cookbook/recipe/0009-book-1/manifest.json"),
                  ("Europeana API", "https://api.europeana.eu/record/v2/search.json?wskey=api2demo&query=*&rows=0")]:
    try:
        check(name, requests.get(url, timeout=15).ok)
    except Exception as e:
        check(name, False, f"({type(e).__name__})")

import os
candidates = [Path(os.environ["DSI_WORKSHOP_DIR"])] if os.environ.get("DSI_WORKSHOP_DIR") else []
candidates += [Path.home() / "shared" / "DSI-Workshop", Path.home() / "data", Path(".").resolve().parent / "data" / "DSI-Workshop"]
found = next((d for d in candidates if (d / "metadata.csv").exists()), None)
check("dataset folder (semantic search)", found is not None, str(found or candidates[0]))

In [2]:
# Setup: run this cell first.
from iiif_helpers import *

<div style="border-left:14px solid #BC8F8F;background:#ffffff;padding:70px 40px;border-radius:4px"><div style="font:600 20px sans-serif;color:#BC8F8F;letter-spacing:.08em">IIIF</div><div style="font:700 48px sans-serif;margin-top:10px;color:#1f2430">Images across institutions</div><div style="font:22px sans-serif;margin-top:16px;color:#1f2430"></div></div>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 2 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Intro: The Terapixel Panorama

The [Terapixel panorama](https://terapixelpanorama.ch/) of the Battle of Murten is an impressive demonstration of the IHiif capabilities

1. Open the panorama: <https://terapixelpanorama.ch/>
2. Open the **developer tools**: press **F12** (or **Ctrl+Shift+I**; on a Mac **Cmd+Option+I**).
4. Filter by **Img** (in Firefox: **Images**).
5. **Pan and zoom** in the panorama and watch the list fill up.
6. Click one request. The **Headers** tab shows its URL, the **Preview** tab the picture that came back.

In [5]:
url_panel("Terapixel panorama, Battle of Murten", [
    ("Terapixel panorama", "https://terapixelpanorama.ch/", "Open it, press F12, Network tab, filter by Img, then pan and zoom."),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 3 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## What is IIIF?

**International Image Interoperability Framework**

- A shared, open **standard** for delivering image-based resources on the web (and audio, video and 3D), so that they work across institutions.
- Also an open, global **community** of libraries, archives, museums, universities, developers and researchers who develop the specifications, build them into software and publish their collections.
- In short: images and their descriptions are offered through open interfaces (**APIs**), and any compatible tool can use them.



<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 4 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Why does it matter?

![IIIF connects institutions, through open APIs, to any viewer or tool](../misc/assets/img/iiif_why.svg)

- The images **stay at the institution**. Nothing is copied or re-hosted.
- **One viewer** can show objects from many places side by side.
- Anyone can compare, annotate and reuse objects, because the interfaces are the same everywhere.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 5 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Where do I find IIIF?

<img src="../misc/assets/img/iiif_logo.png" alt="IIIF logo" align="right" width="130">

On many publication platforms. Some show this **IIIF logo**, or a link called **IIIF manifest**, on the object page. Others hide it in a menu, or do not show it at all.

- [e-rara](https://www.e-rara.ch/) and [e-codices](https://www.e-codices.ch/): Swiss rare prints and manuscripts
- [e-manuscripta](https://www.e-manuscripta.ch/): manuscripts and archival material
- [Gallica](https://gallica.bnf.fr/): Bibliothèque nationale de France
- [Europeana](https://www.europeana.eu/): the European aggregator (next notebook)
- [Biblissima](https://portail.biblissima.fr/): a portal for medieval and early modern sources

<small>IIIF logo: International Image Interoperability Framework, public domain (text logo), via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:International_Image_Interoperability_Framework_logo.png).</small>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 6 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## A Book of Hours with many owners

[Fragmentarium](https://fragmentarium.ms/) record F-mp8l is a *virtual reconstruction* of a Book of Hours (France, early 15th century) that was taken apart in the 1940s. Its leaves are held by many institutions: Yale, Kent, Cleveland Public Library and more.

One manifest brings them back together: **61 canvases**, each with an image that stays on its owner's server, and **16 ranges** that rebuild the structure of the book (the Hours of the Virgin, the Hours of the Holy Spirit, ...).

Open the **raw JSON** of the manifest in a browser tab (below) and unfold it: `sequences` → `canvases` → one canvas → `images` → `resource` → `service`. The next slide is the same tree, as a diagram.

In [8]:
BOH_MANIFEST = "https://fragmentarium.ms/metadata/iiif/F-mp8l/manifest.json"

url_panel("A Book of Hours, reassembled", [
    ("Fragmentarium record (reconstruction)", "https://fragmentarium.ms/overview/F-mp8l"),
    ("IIIF manifest = the raw JSON", BOH_MANIFEST, "Open it in a browser tab (Firefox shows JSON as an unfoldable tree). Or paste it into Mirador: Add resource."),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 7 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

### How a manifest is nested

![IIIF structure: collection, manifest, ranges, canvases, images](../misc/assets/img/iiif_structure.svg)

- <b style="color:rosybrown">Manifest</b>: one object, one URL. In the Book of Hours: the whole reconstruction.
- <b style="color:crimson">Canvas</b>: a virtual page with width and height. List order is page order. Here: one leaf.
- <b style="color:blueviolet">Image</b>: attached to a canvas by a `painting` annotation. It can be hosted anywhere: every leaf on its owner's server.
- **Range**: a section (a table of contents) that groups canvases *without* reordering them.

In [11]:
# The same tree as in the browser, printed: keys, list lengths, the first element of each list.
boh = get_json(BOH_MANIFEST)
show_outline(boh)

manifest   (sc:Manifest)
  sequences   [1 items]
    [0]   (sc:Sequence)
      canvases   [61 items]
        [0]   (sc:Canvas)
          images   [1 items]
            [0]   (oa:Annotation)
              resource   (dctypes:Image)
                service
  structures   [16 items]
    [0]   (sc:Range)
      ranges   [5 items]


In [12]:
# An aggregator can generate a manifest for an institution that offers none: Europeana, for an item of the Museum Kunstpalast (Düsseldorf).
EUROPEANA_MANIFEST = "https://iiif.europeana.eu/presentation/2064114/item_TATJZ2EO2DPKQKYJYJ53YATT6KYU62LS/manifest"
show_outline(get_json(EUROPEANA_MANIFEST))

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 8 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Annotations

A page of a digitised book can carry more than its image. On e-rara and e-manuscripta, a page has a **text layer**: a transcription, stored separately from the image and pointing at it. That is the idea of an **annotation**: a note that is about a page (or a part of it) without changing it.

This manuscript is one example: <https://www.e-rara.ch/bes_1/content/pagetext/32643449>

**The story**

- **The woman:** [Juliane von Krüdener](https://de.wikipedia.org/wiki/Juliane_von_Kr%C3%BCdener) (1764–1824), born in Riga into the Baltic German nobility, married at 17 to a Russian diplomat. After a religious conversion in 1804 she became a Pietist preacher and *prophetess*, travelled through south-west Germany, Alsace and Switzerland, and gained influence even over Tsar Alexander I. A noblewoman preaching in public: this shocked her contemporaries.
- **Her fate:** she was expelled from one state after another (Württemberg, Baden, Basel) and ended up in Russia, where she died in 1824 in Crimea. Some thought the treatment too harsh.
- **The man:** Friedrich von Dillenburg defends her in this manuscript.
- **Why it is here:** the defence is *text about a person*, preserved with the page images. Reading it takes a human. What would a program do with it?


In [14]:
url_panel("Text layers on e-rara and e-manuscripta", [
    ("The annotated manuscript (page text, e-rara)", "https://www.e-rara.ch/bes_1/content/pagetext/32643449", "The transcription of the page, separate from the image."),
    ("Another page text: a handwritten note by Emmi Ball-Hennings (e-manuscripta, Swiss National Library)", "https://www.e-manuscripta.ch/snl/content/pagetext/4317519", "Open it to see the transcription as plain text."),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 9 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## How it works behind the scenes

Two specifications do the work, and each has its own kind of URL.

| API | Question it answers | What you get |
|---|---|---|
| **Presentation API** | What is this object, and how is it organised? | A JSON *manifest* |
| **Image API** | Give me this image, in this form. | Pixels, selected by a URL |

Below: the manifest of a small book (five pages, BnF / Gallica, from the IIIF Cookbook), and the URL of one image from it. Open both in a browser tab. The manifest contains the base URL of the image server; the image URL is built from it.


In [15]:
BOOK_URL = "https://iiif.io/api/cookbook/recipe/0009-book-1/manifest.json"
book = get_json(BOOK_URL)
PAGE = book["items"][2]                   # the title page
_, SERVICE = canvas_image(PAGE)
info = get_json(f"{SERVICE}/info.json")   # what the server can do
W, H = info["width"], info["height"]

url_panel("One object, one image", [
    ("Presentation API: the manifest (JSON)", BOOK_URL, "Open it in a browser tab: title, canvases in order, and for each the address of its image."),
    ("Image API: the title page as an image", iiif_url(SERVICE, size="600,"), "The counterpart: pixels, delivered by a different server."),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 10 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## One image, any view

An image service answers URLs of a fixed form:

```
{service}/{region}/{size}/{rotation}/{quality}.{format}
```

![Anatomy of an IIIF image URL](../misc/assets/img/iiif_image_url.svg)

In [16]:
show_views(SERVICE, [
    ("full", "full", "260,", "0", "default"),
    ("a region", f"{W*28//100},{H*45//100},{W*33//100},{H*23//100}", "260,", "0", "default"),
    ("rotated 90°", "full", ",260", "90", "default"),
    ("grey", "full", "260,", "0", "gray"),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 11 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Changing how an image is displayed by editing its URL

*Optional, in your own browser, at your own pace.*

Every view of an image is just a URL. Open the image below in a browser tab, then change one segment of the address in the address bar and press Enter.

![Anatomy of an IIIF image URL](../misc/assets/img/iiif_image_url.svg)

- <b style="color:rosybrown">region</b>: `full`, `square`, `x,y,w,h` (pixels), `pct:x,y,w,h` (percent)
- <b style="color:coral">size</b>: `max`, `400,` (width), `,300` (height), `!400,300` (fit inside), `pct:50`
- <b style="color:crimson">rotation</b>: `0`, `90`, `180`, `270`, `!0` (mirrored)
- <b style="color:blueviolet">quality</b> and <b style="color:blueviolet">format</b>: `default`, `gray`, `bitonal`; `.jpg`, `.png`


In [18]:
EDIT_SERVICE = "https://apsida.cut.ac.cy/iiif-img/34769"   # image service of https://apsida.cut.ac.cy/iiif/36404/manifest
url_panel("An image from Apsida (Cyprus University of Technology)", [
    ("Image URL", iiif_url(EDIT_SERVICE, size="600,"), "Open it in a browser tab, then edit the segments in the address bar."),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 12 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## How do I use it?

1. Find the **manifest link** (or the IIIF logo) on the object page.
2. **Copy** the link.
3. Open a IIIF viewer, for example [Mirador](https://projectmirador.org/demo/) or [Universal Viewer](https://universalviewer.io/).
4. **Paste** the link. The object appears in the viewer.

In Mirador you can place several objects side by side to compare them, and a workspace can be exported and imported again. Many IIIF tools work with images without downloading and uploading them: for example [Tropy](https://tropy.org/) for archival material, or eScriptorium and Transkribus for text recognition. A long list: <https://iiif.io/get-started/tools/>.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 13 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Mirador: paste in a manifest

[Mirador](https://projectmirador.org/) is given only the manifest URL. It works out page order, thumbnails, deep zoom and the table of contents by itself.

**Your turn** *(about 5 minutes)*

1. Open the **[Mirador demo](https://projectmirador.org/demo/)**.
2. Click **Add resource**, paste one of the two manifest URLs below, and press **Add**. Add the second one too and place them side by side.
3. Then find a manifest of your own: open one of the sites from *Where do I find IIIF?*, pick an object, and look for the IIIF logo or a link called *IIIF* or *Manifest*, often in a share, download or export menu.

**Example manifests** *(copy the link)*

- Watercolour *0m* by Lucius Burckhardt, e-manuscripta:
  `https://www.e-manuscripta.ch/i3f/v20/4602948/manifest`
- *Der Wanderer über dem Nebelmeer*, Deutsche Fotothek via arthistoricum.net:
  `https://iiif.arthistoricum.net/proxy/fotothek/df_hauptkatalog_0182654/manifest.json`

<small>**Credits.**
*0m* / von Lucius Burckhardt, [S.l.], [between 1925 and 2003]. Watercolour on paper, 23 x 31 cm. Universitätsbibliothek Basel, UBH NL 351 : P:792 (Nachlass Lucius Burckhardt and Annemarie Burckhardt-Wackernagel). Via [e-manuscripta](https://www.e-manuscripta.ch/), DOI [10.7891/e-manuscripta-170584](https://doi.org/10.7891/e-manuscripta-170584).<br>
© Deutsche Fotothek / Karin Stein. License: all rights reserved. Higher-resolution copies, prints and usage rights can be obtained from the [Deutsche Fotothek](https://www.deutschefotothek.de/).</small>

<div style="border-left:14px solid #FF7F50;background:#ffffff;padding:70px 40px;border-radius:4px"><div style="font:600 20px sans-serif;color:#FF7F50;letter-spacing:.08em">EUROPEANA</div><div style="font:700 48px sans-serif;margin-top:10px;color:#1f2430">Distant Reading Europeana</div><div style="font:22px sans-serif;margin-top:16px;color:#1f2430">Who contributes and what is in it?</div></div>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 1 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

# Europeana

The IIIF notebook showed what IIIF does *within one platform*. But there are thousands of libraries, archives and museums, each with its own platform. How does a researcher find objects across all of them?

The usual answer is an **aggregator**: a service that collects descriptions of objects from many institutions into one searchable index. [Europeana](https://www.europeana.eu/) is the largest for European cultural heritage.

In [20]:
# Setup: run this cell first.
from europeana_helpers import *
from iiif_helpers import url_panel, show_mirador

print(API_KEY_STATUS)
overview, institutions, aggregators, countries = load_overview()   # cached counts; fetched once if missing
total_items = overview["total_items"]

Using the shared demo key (limited requests). For your own free key see https://pro.europeana.eu/page/get-api and save it to notebooks/api-key-europeana.txt
Loaded cached counts from 2026-11-09
3,948 institutions (data providers), 123 providers / aggregators, 47 countries


<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 2 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## How data gets to Europeana

![Europeana flow: institutions reach Europeana directly, through aggregators, or through projects](../misc/assets/img/europeana_flow.svg)

- Institutions run their own **platforms** and offer **metadata** (for harvesting) and, sometimes, **IIIF** (for images).
- Europeana **harvests the metadata** into its own format and reaches institutions *directly*, *via an aggregator* or *via a project*.
- Europeana is mainly an **index of descriptions**. The full-resolution images stay with the institutions.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 3 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## The Europeana API

An **API** lets a program send a request and receive structured data (JSON) instead of a web page. We use the **Search API**, which can return items, or only *counts* per value of a field.

| API | Purpose |
|---|---|
| **Search API** | Query and filter the whole index; counts per value ("facets") |
| **Record API** | Full metadata of one item |

The whole index, counted by media type (`rows=0`: no items, only counts):

In [21]:
print(f"Items in the index: {total_items:,}\n")
for t, n in facet_counts("TYPE"):
    print(f"  {t:<6} {n:>12,}   {n / total_items:6.1%}")

Items in the index: 63,089,721

  IMAGE    36,585,819    58.0%
  TEXT     24,856,491    39.4%
  SOUND     1,244,629     2.0%
  VIDEO       389,152     0.6%
  3D           13,630     0.0%


<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 4 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Who contributes?

Counting items per institution (`DATA_PROVIDER`). The bar shows each institution's mix of media types.

In [22]:
show_table(institutions, sort_by="total", n=12, art=True, iiif=True)

Institution,Total,Image,Text,Sound,Video,3D,Art theme,IIIF-linked,Media mix
Naturalis Biodiversity Center,"5,045,088","4,379,766",907,"664,130",285,–,"3,832",907,
National Library of France,"2,999,311","573,610","2,372,854","52,846",–,1,"113,231","1,886,125",
"The Trustees of the Natural History Museum, London","2,462,902","2,462,902",–,–,–,–,"10,207",0,
Bavarian State Library,"2,176,001",–,"2,176,001",–,–,–,"5,506",0,
Virtual Library of Historical Press,"1,735,320",–,"1,735,320",–,–,–,5,0,
Swedish National Heritage Board,"1,414,618","304,112","1,110,506",–,–,–,"10,219",0,
Meise Botanic Garden,"1,384,329","1,383,746",583,–,–,–,"4,076",70,
Deutsche Fotothek,"1,314,205","1,314,205",–,–,–,–,"142,760",0,
"KB, National Library of the Netherlands","987,236","16,250","970,986",–,–,–,"7,838","437,593",
Royal Botanic Garden Edinburgh,"858,745","858,720",25,–,–,–,402,"858,745",


<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 5 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Routes and countries

The `PROVIDER` facet shows the route by which items arrived: national aggregators such as the German Digital Library (DDB), or projects.

In [23]:
show_table(aggregators, sort_by="total", n=8, first_col="Provider / aggregator")
show_table(countries, sort_by="total", n=8, first_col="Country")

Provider / aggregator,Total,Image,Text,Sound,Video,3D,Media mix
OpenUp!,"12,276,717","11,308,085","284,943","683,169",318,202,
German Digital Library,"6,924,833","3,249,063","3,659,778","15,206",730,56,
Swedish Open Cultural Heritage,"5,335,192","4,536,110","797,332","1,598",148,4,
The European Library,"4,828,348","175,589","4,652,454",208,97,–,
Hispana,"4,579,734","759,449","3,807,191","2,149","10,916",29,
Digital Libraries Federation,"2,889,913","98,512","2,789,098","1,431",869,3,
CARARE,"2,672,162","1,491,100","1,171,866",–,"1,396","7,800",
Dutch Collections for Europe,"2,029,769","2,007,771","18,554",16,"3,426",2,


Country,Total,Image,Text,Sound,Video,3D,Media mix
Netherlands,"9,707,787","7,680,173","1,151,129","862,084","14,057",344,
Germany,"8,968,202","4,427,837","4,435,320","54,089","50,876",80,
Spain,"6,634,833","1,445,022","5,072,408","8,515","106,983","1,905",
Sweden,"6,078,207","4,635,215","1,441,129","1,598",261,4,
United Kingdom,"5,660,269","5,020,265","584,109","45,305","9,721",869,
France,"4,724,898","2,136,120","2,443,572","97,407","47,137",662,
Poland,"3,783,073","303,668","3,461,620","5,063","12,026",696,
Belgium,"2,353,172","2,270,159","80,553",–,"1,836",624,


<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 6 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## A closer look at one institution

The table says how much an institution holds, not what. A few more facet requests restricted to one institution (`qf=DATA_PROVIDER:"name"`) give a quick profile.

In [24]:
institution_preview("Wellcome Collection")

IMAGE,"90,718",
by 4.0,"84,755",
by-nc 4.0,"5,431",
zero 1.0,525,
by-nc-nd 4.0,7,
1st c.,3,
15th c.,7,
16th c.,196,
17th c.,815,
18th c.,"3,761",
19th c.,"9,968",


<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 7 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Beyond the website's search box

The website filters well. The API is more useful for *numbers across the whole index* and *stacked filters*, including ones the website does not offer, such as a range on the year, or "records that name a IIIF manifest". Each step narrows the previous one:

In [25]:
steps = [
    ("query: ship",                                   dict(query="ship")),
    ("+ images only",                                 dict(query="ship", qf=["TYPE:IMAGE"])),
    ("+ openly reusable",                             dict(query="ship", qf=["TYPE:IMAGE"], reusability="open")),
    ("+ dated 1800 to 1899",                          dict(query="ship", qf=["TYPE:IMAGE", "YEAR:[1800 TO 1899]"], reusability="open")),
    ("+ record names a IIIF manifest (lower bound)", dict(query="ship", qf=["TYPE:IMAGE", "YEAR:[1800 TO 1899]", "wr_dcterms_isReferencedBy:*"], reusability="open")),
]
for label, kw in steps:
    print(f"{label:<44} {count(**kw):>10,}")

query: ship                                     151,637
+ images only                                   122,494
+ openly reusable                                73,762
+ dated 1800 to 1899                              1,031
+ record names a IIIF manifest (lower bound)         24


<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 8 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## What one record contains

| Field | Meaning |
|---|---|
| `dataProvider` | the institution that holds the object |
| `provider` | the aggregator or route |
| `edmIsShownAt` | the object's page on the institution's website |
| `edmIsShownBy` | the media file on the institution's server |
| `edmPreview` | a small preview image served by Europeana |
| `rights` | the rights statement |

In [ ]:
sample_results = search_europeana(query="portrait", rows=20, qf=["TYPE:IMAGE"], theme="art", reusability="open")
sample = sample_results["items"]
item = sample[0]
for field in ["title", "dataProvider", "provider", "country", "type", "rights", "edmIsShownAt", "edmIsShownBy", "edmPreview"]:
    value = item.get(field)
    if isinstance(value, list):
        value = value[0]
    print(f"{field:<14} {value}")

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 9 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## From a record to a manifest

![From a Europeana item page to its IIIF manifest](../misc/assets/img/from_europeana_img_to_manifest.webp)

- Europeana does not hold the images. `edmIsShownBy` points to the file on the **institution's own server**.
- Many of those institutions have **not implemented IIIF**. Europeana generates a manifest for every record anyway, from the record's metadata and image link: the dataset ID and local ID are copied unchanged into a new address.
- So the manifest comes from the *aggregator*, not from the institution. The canvases and labels are built by Europeana; the image stays where it is.
- The limit: if the institution offers no IIIF *image* service, the manifest can only point to the plain file. Below, a record from the Museum Kunstpalast (Düsseldorf), whose own website shows no manifest.


In [ ]:
url_panel("Museum Kunstpalast, Düsseldorf: a drawing, via Europeana", [
    ("Item page on Europeana", "https://www.europeana.eu/en/item/2064114/item_TATJZ2EO2DPKQKYJYJ53YATT6KYU62LS", "The record. The institution's own page has no IIIF link."),
    ("Manifest generated by Europeana", "https://iiif.europeana.eu/presentation/2064114/item_TATJZ2EO2DPKQKYJYJ53YATT6KYU62LS/manifest", "Same IDs, new domain, /manifest appended. Open it, or paste it into Mirador."),
    ("Image on the institution's server", "https://emuseum.duesseldorf.de/internal/media/dispatcher/324917/full", "A plain JPEG: the manifest has no image service for it, so editing this URL changes nothing."),
])

# Notebook 3: Semantic search via Embeddings

The Europeana section of the intro notebook found objects through their *metadata*. Now we will find results by comparing **Embeddings**. The 
[CLIP](https://openai.com/research/clip) Model computes embeddings for images *and* short texts in the same space, so a sentence can be compared directly with every image.

## CLIP: text and images in one space

![CLIP contrastive pre-training: a text encoder and an image encoder produce vectors that are compared in a matrix](../misc/assets/img/clip.svg)

<small>Contrastive pre-training of CLIP: matching image-text pairs are pulled together (blue diagonal), all other pairs pushed apart. Figure: OpenAI, ["CLIP: Connecting text and images"](https://openai.com/index/clip/) (Radford et al., 2021).</small>

The image half was applied to the whole collection beforehand. For a query only the **text encoder** runs, which is fast on a laptop.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 1 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

In [41]:
# Setup: run this cell first.
import numpy as np
from IPython.display import display
from clip_helpers import *
from region_helpers import *
from iiif_helpers import show_mirador_many, url_panel

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 3 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## The collection

`metadata.csv` and the vectors are in the same order: row *i* describes vector *i*. A random sample (the images are fetched from their institutions now):

In [ ]:
coll = Collection()
coll.summary()

rng = np.random.default_rng(1)
coll.show_grid(rng.choice(len(coll.meta), 12, replace=False))
coll.top_institutions_and_countries()
clip_model = Clip()
coll.top_institutions_and_countries()


<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 3 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Search with Europeana keywords

The Europeana search of the previous notebook looks for **words** in the metadata: title, description, creator, subject. Here a keyword search for *Amor and Psyche*, restricted to open images of the Art collection. Keep the result in mind: it is the comparison for the next section.

In [ ]:
import europeana_helpers as eu

KEYWORD = "Amor and Psyche"
result = eu.search_europeana(query=KEYWORD, rows=12, qf=["TYPE:IMAGE"], theme="art", reusability="open")
print(f"{result['totalResults']} records contain these words. The first 12:")
eu.show_results(result["items"])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 6 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Search with a Text Embedding

The query becomes a vector in the same 512-dimensional space. Both are normalised to length 1, so their **dot product** is their cosine similarity: 1 means the same direction, near 0 unrelated. Ranking the whole collection is one matrix–vector product.

In [ ]:
QUERY = "Amor and Psyche"

scores = coll.similarity(clip_model.text(QUERY))
best = coll.top_k(scores, 12)
coll.show_grid(best, scores)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 7 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Restricting the search with a mask

Because the metadata is aligned with the vectors, a filter is just a boolean **mask** applied before ranking. Here: only images dated before 1900.

In [ ]:
QUERY = "a portrait of a person"
mask = (coll.year > 0) & (coll.year < 1900)                          # dated, and before 1900
# mask = np.isin(coll.country, ["Sweden", "Finland", "Norway"])      # alternative: by country
# mask = np.isin(coll.institution, ["Wellcome Collection"])          # alternative: by institution

scores = coll.similarity(clip_model.text(QUERY))
print(f"{mask.sum():,} of {len(mask):,} images pass the filter. Top results for {QUERY!r}:\n")
coll.show_grid(coll.top_k(scores, 12, mask), scores)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 8 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Search with an image

The same space holds image embeddings, so an image can be the query: "more like this". **Upload an image** of your own (a photograph, a scan, a screenshot), then run the cell below it. Without an upload, the cell uses the best match of the text search above.

In [ ]:
import io
import ipywidgets as widgets
from PIL import Image

upload = widgets.FileUpload(accept="image/*", multiple=False, description="Upload an image")
display(upload)

In [ ]:
files = list(upload.value.values()) if isinstance(upload.value, dict) else list(upload.value)

if files:                                  # your own image
    query_image = Image.open(io.BytesIO(bytes(files[0]["content"]))).convert("RGB")
    print("Query: your image")
    display(query_image.copy().resize((200, int(200 * query_image.height / query_image.width))))
    scores = coll.similarity(clip_model.image(query_image))
    coll.show_grid(coll.top_k(scores, 12), scores)
else:                                      # no upload: more like the best match of the text search
    SEED = int(best[0])
    print("Query: collection image", SEED, "-", coll.meta[SEED]["title"][:60])
    display(coll.small_image(SEED))
    scores = coll.similarity(coll.V[SEED].astype(np.float32))
    scores[SEED] = -1                      # do not return the query itself
    coll.show_grid(coll.top_k(scores, 12), scores)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 9 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## One picture, five regions

An image is rarely about one thing. We cut it into **regions**, embed each region on its own, and search the collection with it.

A Swedish advertising stamp (*reklammärke*) of 1939 for *Ljusäkta Fiona Färger*, from the Malmö Museum ([record](http://carlotta.malmo.se/carlotta-mmus/web/object/178916), CC BY 4.0). A kneeling figure by candlelight, a snake beside it, a band of coloured stripes below. We search with the whole stamp, the figure, only the face, one foot, the snake, and the motif without the perforated edge (the "teeth").

This record has no IIIF image service and its original file is not available, so the regions are cut from the small version (400 px) on your computer: coarse, which matters for small regions. `frac(x, y, w, h)` gives a region as fractions of the picture (0 to 1), converted to pixels.

The stamp is tall. With `fit="crop"` CLIP reads the central square, as it did for the vectors of the collection; the default `"pad"` would add grey bars and the search would follow the grey (try it).

In [ ]:
ID = "/91658/MM_objekt_178916"      # a record of the collection (Europeana ID)
row = next(i for i, r in enumerate(coll.meta) if r["id"] == ID)
pic = Picture(row, coll)

regions = {
    "kneeling figure":  pic.frac(0.20, 0.19, 0.50, 0.54),
    "face":             pic.frac(0.37, 0.20, 0.22, 0.17),
    "foot":             pic.frac(0.34, 0.59, 0.14, 0.13),
    "snake":            pic.frac(0.69, 0.54, 0.19, 0.20),
    "motif (no teeth)": pic.frac(0.055, 0.04, 0.87, 0.915),
}
pic.show(regions)

The same search with seven queries: the whole stamp and the five regions. Each row is one query, each card one of the 8 most similar images of the collection (the stamp itself is left out).

In [ ]:
exp = Experiment(coll, clip_model, pic)
exp.add("whole stamp", fit="crop")
for name, region in regions.items():
    exp.add(name, region, fit="crop")

exp.run(k=8)
exp.show()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 10 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Colour, grey, black and white

The IIIF **quality** is part of the URL, so the same region can be requested as `default`, `gray` or `bitonal` (pure black and white). Does the model still recognise it? A scan from a bitonal microfilm, or a photocopy, is such an image.

In [ ]:
q = Experiment(coll, clip_model, pic)
for quality in ("default", "gray", "bitonal"):
    q.add(f"whole, {quality}", quality=quality, fit="crop")
    q.add(f"face, {quality}", regions["face"], quality=quality, fit="crop")

q.run(k=8)
q.show()
q.overlap()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 11 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Regions against text

A text query lives in the same space, so it can be added as one more row: does the region find what the *words* find? Compare the snake region with the word "snake".

In [ ]:
exp.add("text: snake", text="a snake")
exp.add("text: reading", text="a person reading by candlelight")
exp.run(k=8)
exp.show()
exp.overlap()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 12 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Cut everything: tiles

Instead of choosing regions by hand, cut the whole picture into overlapping **tiles**, embed every tile, and search the tiles with a text. The result is a map of where in the picture something is. With IIIF each tile is one small request, so even a very large picture is possible (you will find large pictures in Part B).

In [ ]:
scan = TileScan(pic, clip_model, rows=4, overlap=0.5)
found = scan.show("a snake", k=3)

In [ ]:
other = scan.show("a person", k=3)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 13 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Annotations: pointing at regions

Every region becomes a **web annotation** that targets the canvas with `#xywh=`. The manifest also lists the images found for each region as canvases, grouped in one range per region. Open it in Mirador: the annotations are in the side panel, the images in the table of contents.

In [ ]:
notes = [
    note(regions["kneeling figure"],  "The kneeling figure. Dark painted figures and portraits: no stamp among the results.", hits=exp.top("kneeling figure", 3)),
    note(regions["face"],             "Only the face. Painted portraits of faces and children.", hits=exp.top("face", 3)),
    note(regions["snake"],            "The snake. The region finds a sabre and a signature, the word 'snake' finds snakes.", hits=exp.top("snake", 3)),
]
notes += [note(r, f"a snake? (tile score {s:.2f})") for r, s in found[:2]]

manifest = build_manifest("Demo: a stamp and its details", pic, notes, coll, summary="Regions of one advertising stamp and the images found for them.")
show_manifest(manifest)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 14 / 14</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## The presentation

A `Deck` collects slides from your own results: text, the picture with its regions, the comparison, found images, and the annotated viewer. It is one HTML file (arrow keys to move).

In [ ]:
deck = Deck("Demo: a stamp and its details", author="Workshop", subtitle="Whole image vs. region")
deck.add_text("Question", "Does the collection answer a detail differently from the whole picture?")
deck.add_picture("Picture and regions", pic, regions)
deck.add_compare("Whole stamp vs. regions", exp, text="- The whole stamp finds its own series.\n- The motif without the teeth finds paintings of people reading by candlelight.\n- The snake and the foot are too small: arbitrary results.")
deck.add_mirador("Annotated", manifest)
deck.show()

<div style="border-left:14px solid #2E8B57;background:#ffffff;padding:70px 40px;border-radius:4px"><div style="font:600 20px sans-serif;color:#2E8B57;letter-spacing:.08em">FINAL TASK</div><div style="font:700 48px sans-serif;margin-top:10px;color:#1f2430">Show what you found</div><div style="font:22px sans-serif;margin-top:16px;color:#1f2430">Two minutes each. Then the presentations, and we conclude the workshop.</div></div>

## The task

1. **Choose what you want to show.** The pictures you saved with **Keep for my presentation** (they are in the folder `my_images/` next to the notebooks), a screenshot, or anything you like: an `.html` page, an exported viewer, a few images.
2. **Put it in the shared Switch drive folder** (link below), in a **folder with your name**.
3. **Prepare to speak for 2 minutes:** what you found, and how it might (or might not) relate to one of your own projects.

The files are opened from the drive for you while you speak. Say what you see, including where the method did not work: that is a finding too.

## The Switch drive folder

Run the cell below. The link opens the folder in a new tab.

In [ ]:
import ipywidgets as widgets

DRIVE_URL = "https://drive.switch.ch/index.php/s/JeirWKGtP2HBxEm"

widgets.HTML(
    f"<div style='font:16px sans-serif;padding:14px 18px;border-left:6px solid #2E8B57;background:#f4faf6'>"
    f"<b>Drop your files here:</b><br>"
    f"<a href='{DRIVE_URL}' target='_blank' style='font-size:18px'>{DRIVE_URL}</a>"
    f"</div>"
)

## Your kept pictures

Pictures you kept in the earlier notebooks are listed here. To hand one in, download it first: in the file browser on the left (folder icon) open `DH-Workshop/notebooks/my_images/`, right-click the file, **Download**. Or run the cell after the list: it downloads all kept pictures as one zip.

In [ ]:
from keep_helpers import library

kept = library()
rows = "".join(f"<li><code>{name}</code> {caption}</li>" for name, caption in kept)
widgets.HTML(
    f"<div style='font:14px sans-serif'><b>{len(kept)} kept picture(s)</b> in <code>my_images/</code>"
    + (f"<ul>{rows}</ul>" if kept else ". None yet: go back to a notebook and click <i>Keep for my presentation</i> under a result, or just take a screenshot.")
    + "</div>"
)

In [ ]:
import shutil
from google.colab import files
from keep_helpers import MY_IMAGES

if MY_IMAGES.exists() and any(MY_IMAGES.glob("*.jpg")):
    files.download(shutil.make_archive("/content/my_images", "zip", MY_IMAGES))      # one zip with all kept pictures
else:
    print("No kept pictures yet.")

## Handing in, step by step

1. Download the files you want to show (a screenshot of your own is fine too).
2. Open the Switch drive link above.
3. Create a **folder with your name** and drop your files in.
4. Prepare your 2 minutes. A suggestion: **what you searched** (30 s), **what you found** (60 s), **how it might relate to one of your projects** (30 s).

Then we listen to each other, and conclude the workshop.